# Working with the Overture extracts

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kentstephen/bias-bounty-map-tutorial/blob/main/bias-bounty-explore-tutorial.ipynb)


Everything in this notebook runs against the live, public bucket. Nothing is downloaded to disk,
no credentials, no signup.

**Bias Bounties @ Scale** ships 13 layers for each of four US regions. Five are Overture,
the data being scored:

| layer | Overture theme / type | geometry |
|---|---|---|
| `overture-buildings` | `buildings` / `building` | polygons |
| `overture-roads` | `transportation` / `segment` (`subtype=road`) | lines |
| `overture-rail` | `transportation` / `segment` (`subtype=rail`) | lines |
| `overture-infrastructure` | `base` / `infrastructure` (stations, airports) | mixed |
| `overture-pois` | `places` / `place` | points |

The other eight are the reference layers you score Overture *against*:

| layer | source | geometry | what it is |
|---|---|---|---|
| `microsoft-buildings` | Microsoft GlobalML Building Footprints | polygons | reference baseline for the building coverage gap |
| `census-tiger-roads` | Census TIGER/Line | lines | reference baseline for the road gap |
| `census-acs-housing` | Census ACS 5-Year (B25001) | tract polygons | total housing units per tract |
| `census-cbp` | Census County Business Patterns | tract polygons | business establishment counts per tract |
| `hifld-hospitals` | USGS National Map structures | points | hospitals |
| `hifld-fire-stations` | USGS National Map structures | points | fire stations |
| `hifld-ems-stations` | USGS National Map structures | points | EMS / ambulance services |
| `hifld-schools` | USGS National Map structures | points | schools |

The path is always the same:

```
.../geoparquet/<region>/<region>-<layer>.parquet
```

Three things this notebook covers, in order:

1. **Opening the files** (DuckDB and GeoPandas, one line each).
2. **The nested columns.** `names`, `categories` and `sources` are structs and lists, not strings.
   This is the first thing that bites people.
3. **An interactive map** that reads only the footprints in your current viewport, straight off
   the bucket.

## Setup

### Environment

Every dependency version is locked, in three places that agree:

- `pyproject.toml` + `uv.lock`: run `uv sync`, then open the notebook with `uv run jupyter lab`.
- The PEP 723 block in the next cell: run `uvx juv run bias-bounty-explore-tutorial.ipynb` and juv builds the exact environment on the fly.
- The `uv pip` line in the next cell: on Colab, run the cell once (and restart the runtime if Colab asks).


In [ ]:
# /// script
# requires-python = ">=3.12"
# dependencies = [
#     "arro3-core==0.8.1",
#     "duckdb==1.5.4",
#     "geoarrow-rust-io==0.6.1",
#     "geopandas==1.1.4",
#     "ipywidgets==8.1.8",
#     "lonboard==0.16.0",
#     "matplotlib==3.11.0",
#     "numpy==2.5.1",
#     "obstore==0.11.0",
#     "pandas==3.0.3",
#     "pyarrow==24.0.0",
# ]
# ///
# On Colab this installs the locked versions (uv, because pip takes minutes here).
# find_spec("google") first: find_spec("google.colab") raises off Colab, where no
# "google" package exists at all.
import importlib.util
if importlib.util.find_spec("google") and importlib.util.find_spec("google.colab"):
    %pip install -q uv
    !uv pip install --system -q arro3-core==0.8.1 duckdb==1.5.4 geoarrow-rust-io==0.6.1 geopandas==1.1.4 ipywidgets==8.1.8 lonboard==0.16.0 matplotlib==3.11.0 numpy==2.5.1 obstore==0.11.0 pandas==3.0.3 pyarrow==24.0.0
    # Colab only renders third-party widgets (lonboard is one) after this is enabled.
    from google.colab import output
    output.enable_custom_widget_manager()

In [ ]:
import io, math, warnings
warnings.filterwarnings("ignore")
from concurrent.futures import ThreadPoolExecutor

from IPython.display import display

import duckdb
import geopandas as gpd
import ipywidgets as W
import numpy as np
import obstore
import pandas as pd
from arro3.core import Array, DataType, Table
from geoarrow.rust.io import GeoParquetDataset
from lonboard import Map, PathLayer, PolygonLayer, ScatterplotLayer, SolidPolygonLayer, viz
from lonboard.basemap import CartoStyle, MaplibreBasemap
from lonboard.view_state import MapViewState
from matplotlib import colormaps
from matplotlib.colors import LogNorm
from obstore.store import S3Store
from pyarrow.fs import S3FileSystem

ORG, PRODUCT = "humane-intelligence", "bias-bounty-mapping-equity-challenge"
ROOT = f"{ORG}/{PRODUCT}"
REGIONS = ["maricopa-az", "northern-ca", "eastern-ok", "south-central-tx"]

# Two ways in to the same bytes. DuckDB reads over plain https; obstore/pyarrow use s3://.
HTTPS = f"https://data.source.coop/{ROOT}"

def url(region, layer, base=HTTPS):
    return f"{base}/geoparquet/{region}/{region}-{layer}.parquet"

# The bucket is public and needs no auth. But if you have AWS credentials configured (and
# plenty of people do, for work), pyarrow SIGNS the request, S3 rejects it, and you get a
# baffling ACCESS_DENIED / FileNotFoundError on a public file. An anonymous filesystem is
# immune either way. Note it takes the path WITHOUT the s3:// scheme.
ANON = S3FileSystem(anonymous=True, region="us-west-2")

def s3_path(region, layer):
    return f"us-west-2.opendata.source.coop/{ROOT}/geoparquet/{region}/{region}-{layer}.parquet"

# obstore + geoarrow-rust do the interactive map's viewport reads. Same bucket, same bytes.
store = S3Store("us-west-2.opendata.source.coop", region="us-west-2",
                endpoint="https://s3.us-west-2.amazonaws.com", skip_signature=True)

_objs, _ds = {}, {}

def objects(region):
    if region not in _objs:
        _objs[region] = {
            o["path"].split("/")[-1].removeprefix(f"{region}-").removesuffix(".parquet"): o
            for batch in obstore.list(store, f"{ROOT}/geoparquet/{region}/") for o in batch
        }
    return _objs[region]

def dataset(region, layer):
    if (region, layer) not in _ds:
        _ds[(region, layer)] = GeoParquetDataset.open([objects(region)[layer]], store=store)
    return _ds[(region, layer)]

aois = gpd.read_file(io.BytesIO(bytes(
    obstore.get(store, f"{ROOT}/boundaries/all-aois.geojson").bytes()
)))
print(len(objects("northern-ca")), "layers per region")

### DuckDB

`httpfs` reads the files, `spatial` gives you the `ST_*` functions. Set the S3 region and
path-style addressing on every connection: the bucket name contains dots, so DuckDB's default
virtual-host addressing builds a hostname the TLS certificate doesn't cover.

In [ ]:
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("INSTALL spatial; LOAD spatial;")
con.sql("SET s3_region='us-west-2'; SET s3_url_style='path';")

con.sql(f"SELECT count(*) AS pois FROM '{url('northern-ca', 'overture-pois')}'").show()

### GeoPandas

Pass the `s3://` URI (not `https://`, which pyarrow has no driver for) and a `bbox` to read only
the row groups that intersect it. Here: 4.4M building footprints in the file, ~20k read.

In [ ]:
SAN_ANTONIO = (-98.52, 29.40, -98.46, 29.45)

gdf_buildings = gpd.read_parquet(
    s3_path("south-central-tx", "overture-buildings"), bbox=SAN_ANTONIO, filesystem=ANON,
)
# Or without S3FileSystem and the full path
# path = "s3://us-west-2.opendata.source.coop/humane-intelligence/bias-bounty-mapping-equity-challenge/geoparquet/south-central-tx/south-central-tx-overture-buildings.parquet"
# gdf_buildings = gpd.read_parquet(path, bbox=SAN_ANTONIO)

print(f"{len(gdf_buildings):,} buildings, CRS {gdf_buildings.crs}")
gdf_buildings.plot(figsize=(6, 6), color="#E69F00", linewidth=0)


---
# 1. The nested columns

Overture does not flatten its attributes. `names` and `categories` are **structs**; `sources` is a
**list of structs**. Selecting them raw gives you nested objects, so the useful values need to be
reached into.

In [ ]:
con.sql(f'''
    SELECT column_name, column_type
    FROM (DESCRIBE SELECT names, categories, sources FROM '{url("northern-ca", "overture-pois")}')
''').show(max_width=200)

### `names` and `categories` are structs: use dot access

`names.primary` is the one you want 95% of the time. `names.common` is a `MAP`, so it takes a key
(`['en']`), not an index. Only `names.rules` is a list.

In [ ]:
con.sql(f'''
    SELECT names.primary          AS name,
           names.common['en']     AS name_en,
           categories.primary     AS category,
           categories.alternate   AS also,
           confidence
    FROM '{url("northern-ca", "overture-pois")}'
    WHERE names.primary IS NOT NULL
    LIMIT 8
''').show(max_width=140)

### `sources` is a *list* of structs: index it or unnest it

`sources[1]` is the first source (DuckDB lists are 1-indexed). A feature can have several, so to
count them properly you `unnest`, which explodes one row per source.

In [ ]:
con.sql(f'''
    SELECT names.primary            AS name,
           len(sources)             AS n_sources,
           sources[1].dataset       AS first_dataset,
           sources[1].record_id     AS first_record_id
    FROM '{url("northern-ca", "overture-pois")}'
    WHERE names.primary IS NOT NULL
    LIMIT 8
''').show(max_width=140)

### The same structs in GeoPandas

You do not need DuckDB for this. `gpd.read_parquet` hands the structs back as plain Python
**dicts**, and the lists of structs as **arrays of dicts**. The `.str` accessor indexes both,
despite the name: it works on any object column, not just strings. Chain it to go a level deeper.

In [ ]:
CHICO = (-121.90, 39.68, -121.76, 39.80)

gdf_pois = gpd.read_parquet(
    s3_path("northern-ca", "overture-pois"), bbox=CHICO, filesystem=ANON,
    columns=["names", "categories", "sources", "confidence", "geometry"],
)

gdf_pois["name"]      = gdf_pois["names"].str["primary"]          # struct -> dict
gdf_pois["category"]  = gdf_pois["categories"].str["primary"]
gdf_pois["n_sources"] = gdf_pois["sources"].str.len()             # list of structs -> array of dicts
gdf_pois["first_ds"]  = gdf_pois["sources"].str[0].str["dataset"]  # chain to index into the first one

gdf_pois[["name", "category", "n_sources", "first_ds", "confidence"]].head(8)

`explode` is the pandas equivalent of DuckDB's `unnest`: one row per source. It works on a
GeoDataFrame and keeps the geometry, so you can go straight from this to a per-source map.

In [ ]:
gdf_pois.explode("sources")["sources"].str["dataset"].value_counts()

### Or get typed accessors, with the pyarrow backend

Dict indexing works but it is untyped, and it is slow on big frames. Ask pandas for Arrow-backed
columns instead and you get real `.struct` and `.list` accessors, which stay in Arrow the whole way.

The catch: `dtype_backend` is a **pandas** argument. `gpd.read_parquet` does not accept it, so this
idiom is for the non-spatial columns.

In [ ]:
df_pois = pd.read_parquet(
    s3_path("northern-ca", "overture-pois"), filesystem=ANON,
    columns=["names", "categories", "sources", "confidence"],
    dtype_backend="pyarrow",
)

print("fields on `names`:", df_pois["names"].struct.dtypes.index.tolist())

df_pois.assign(
    name      = df_pois["names"].struct.field("primary"),
    category  = df_pois["categories"].struct.field("primary"),
    n_sources = df_pois["sources"].list.len(),
    first_ds  = df_pois["sources"].list[0].struct.field("dataset"),
)[["name", "category", "n_sources", "first_ds", "confidence"]].head(8)

# The interactive map: draw a box, get the code

Pick a region, tick up to three layers, and they read straight off the bucket. Plus one thing:
**draw a box on the map and it hands you the code to pull exactly those features.**

Hit the **▢ button at the lower-right of the map**, click two corners, and a box is drawn. The
panel below the map then fills with a **complete, copy-pasteable snippet**: the box and every
ticked layer are filled in, DuckDB reads them straight off the live bucket, and lonboard's `viz()`
puts the result back on a map. A **copy snippet** button puts it on the clipboard; **clear**
dismisses it.

`LAYERS` configures every layer in one place; the polygon fills paint on construct via a per-row
colour array (lonboard [#1044](https://github.com/developmentseed/lonboard/issues/1044)).

## Every layer, in one place

One entry per layer. `color` is a flat colour; `ramp` means it is a **choropleth**, coloured by that
column. Everything else is a lonboard trait you can change without touching any other code.

Colours are Okabe-Ito, which stay distinct under colour-vision deficiency. The choropleths use
cividis, a blue-to-yellow ramp, for the same reason.

In [ ]:
MAX_LAYERS = 3      # layers on at once. Each one is its own read and its own GPU buffers.
MINZOOM = 13.0      # below this, the big layers show their row-group boxes instead of features
BIG = 50_000        # rows: over this, a layer is only read for the viewport, never whole
HEAVY = 150_000     # rows on screen, all layers: past this the status line tells you to zoom in

LAYERS = {
    # name                    how it is drawn                                     what it is
    "census-acs-housing": {"ramp": "housing_units", "opacity": 0.85, "line": [40, 40, 40]},
    "census-cbp":         {"ramp": "cbp_estab",     "opacity": 0.85, "line": [40, 40, 40]},

    "hifld-hospitals":    {"color": "#D55E00", "opacity": 1.0,  "radius": 6},
    "hifld-fire-stations":{"color": "#E69F00", "opacity": 1.0,  "radius": 5},
    "hifld-ems-stations": {"color": "#CC79A7", "opacity": 1.0,  "radius": 5},
    "hifld-schools":      {"color": "#56B4E9", "opacity": 1.0,  "radius": 4},

    "overture-buildings": {"color": "#E69F00", "opacity": 0.8},
    "overture-roads":     {"color": "#56B4E9", "opacity": 0.9,  "width": 1.5},
    "overture-rail":      {"color": "#CC79A7", "opacity": 0.9,  "width": 1.5},
    "overture-infrastructure": {"color": "#009E73", "opacity": 0.9, "radius": 4, "width": 1.5},
    "overture-pois":      {"color": "#F0E442", "opacity": 0.9,  "radius": 3},

    "microsoft-buildings":{"color": "#0072B2", "opacity": 0.8},
    "census-tiger-roads": {"color": "#D55E00", "opacity": 0.9,  "width": 1.5},
}

CIVIDIS = colormaps["cividis"]
MISSING = (120, 120, 120)      # a tract with no count is grey, NOT the dark end of the ramp

def cividis_fill(values):
    '''count per tract -> (N, 3) uint8 cividis, log-normed. Log because tract counts are skewed:
    on a linear ramp almost every tract lands in the bottom bucket and the map goes flat.'''
    v = np.asarray(values, dtype="float64")
    finite = v[np.isfinite(v) & (v > 0)]
    norm = LogNorm(vmin=max(finite.min(), 1), vmax=max(finite.max(), 2))
    rgb = (CIVIDIS(norm(np.where(v > 0, v, np.nan)))[:, :3] * 255).astype("uint8")
    rgb[~np.isfinite(v) | (v <= 0)] = MISSING
    return rgb

## Building a layer

Two paths, and the split is deliberate.

A **choropleth** is read whole through GeoPandas and built with `PolygonLayer.from_geopandas`.
These are 591 to 6,012 tracts: small enough to read in full.

Everything **else** is read arrow-native through geoarrow-rust, and only for the viewport when the
layer is big. That is the cloud-native read this whole project is about, and it is what lets you
put a million buildings on a map without downloading a million buildings.

In [ ]:
def choropleth(region, name, cfg):
    '''A census layer: one polygon per tract, filled by its count.'''
    gdf = gpd.read_parquet(s3_path(region, name), filesystem=ANON)
    return [PolygonLayer.from_geopandas(
        gdf,
        get_fill_color=cividis_fill(gdf[cfg["ramp"]]),
        get_line_color=cfg.get("line", [40, 40, 40]),
        line_width_min_pixels=0.5,
        opacity=cfg["opacity"],
        pickable=True,
    )]

# Map each geoarrow geometry type to a layer kind, so we build the right layer class directly.
GEOM = {
    "geoarrow.point":           "point",
    "geoarrow.multipoint":      "point",
    "geoarrow.linestring":      "line",
    "geoarrow.multilinestring": "line",
    "geoarrow.polygon":         "area",
    "geoarrow.multipolygon":    "area",
}

def geom_kind(table):
    md = table.schema.field("geometry").metadata_str
    return GEOM.get(md.get("ARROW:extension:name"))

def one_layer(table, kind, cfg):
    """A table of one geometry type -> its lonboard layer. Every trait comes from cfg."""
    if kind == "point":
        return ScatterplotLayer(table=table, get_fill_color=cfg["color"],
                                radius_min_pixels=cfg.get("radius", 3),
                                opacity=cfg["opacity"], pickable=True)
    if kind == "line":
        return PathLayer(table=table, get_color=cfg["color"],
                         width_min_pixels=cfg.get("width", 1.5),
                         opacity=cfg["opacity"], pickable=True)
    # Overture polygons show as outlines, but picking needs a fill surface to hit-test against.
    # filled=True with a fully transparent fill keeps the outline look while making the whole
    # footprint pickable.
    return PolygonLayer(table=table, filled=True, stroked=True,
                        get_fill_color=[0, 0, 0, 0], get_line_color=cfg["color"],
                        line_width_min_pixels=0.5, opacity=cfg["opacity"], pickable=True,
                        auto_highlight=table.num_rows <= 50_000)

def features(region, name, cfg, bbox):
    """Everything that is not a choropleth. Read the viewport if the layer is big, else the whole file."""
    ds = dataset(region, name)
    table = ds.read(bbox=bbox) if (ds.num_rows > BIG and bbox) else ds.read()
    if table.num_rows == 0:
        return []

    kind = geom_kind(table)
    if kind:
        return [one_layer(table, kind, cfg)]

    # Mixed-geometry layer (overture-infrastructure): use viz() only to split it by geometry type,
    # then build each sub-layer here.
    return [one_layer(l.table, k, cfg)
            for l in viz(table).layers
            if (k := {"ScatterplotLayer": "point", "PathLayer": "line",
                      "PolygonLayer": "area"}.get(type(l).__name__))]

def build(region, name, bbox=None):
    cfg = LAYERS[name]
    return choropleth(region, name, cfg) if "ramp" in cfg else features(region, name, cfg, bbox)

def aoi_layer(region):
    return PolygonLayer.from_geopandas(
        aois[aois.folder == region], filled=False, stroked=True,
        get_line_color=[255, 255, 255], line_width_min_pixels=1.5,
    )

def aoi_bbox(region):
    return tuple(aois[aois.folder == region].total_bounds)

# Below MINZOOM a big layer draws its row-group boxes (the parquet file's own spatial index)
# instead of its features: one footer read, and a viewport read fetches only the boxes it touches.
# Grey, so it reads as file structure rather than data.
BOX_LINE, BOX_FILL = "#B0B0B0", "#B0B0B012"

def partition_boxes(region, name):
    bounds = dataset(region, name).fragments[0].row_groups_bounds()
    n = len(bounds)
    table = Table.from_arrays(
        [bounds, Array([name] * n, DataType.string()), Array(np.arange(n), DataType.int32())],
        names=["geometry", "layer", "row_group"],
    )
    # not pickable: they cover the viewport and would block clicks on the features underneath
    return [PolygonLayer(table=table, get_fill_color=BOX_FILL, get_line_color=BOX_LINE,
                         stroked=True, filled=True, line_width_min_pixels=1, pickable=False)]

def is_big(region, name):
    return "ramp" not in LAYERS[name] and dataset(region, name).num_rows > BIG

## Finding a place

Photon (Komoot's geocoder, OpenStreetMap data, no API key). Type a place, press **Enter**, and
the hits land in a list, biased to the current region's bounding box. Pick one and the map goes
there.

In [ ]:
import json, urllib.parse, urllib.request

PHOTON = "https://photon.komoot.io/api"
JUMP_ZOOM = 14.0     # zoom to land on for a hit with no extent (an address, a shop)

def photon(query, bbox, limit=6):
    '''Geocode `query`, biased to `bbox`. Returns [(label, (lon, lat, extent)), ...] where
    extent is Photon's own bounding box for the feature (towns and counties have one; a single
    address does not), in its [minlon, maxlat, maxlon, minlat] order.'''
    params = urllib.parse.urlencode({
        "q": query,
        "limit": limit,
        "bbox": ",".join(f"{v:.4f}" for v in bbox),
    })
    with urllib.request.urlopen(f"{PHOTON}?{params}", timeout=6) as resp:
        features = json.load(resp)["features"]

    hits, seen = [], {}
    for f in features:
        p = f["properties"]
        # Photon labels vary by feature type: a town has name+state, an address has street+city.
        # dict.fromkeys de-dupes while keeping order ("Chico, Chico, CA" -> "Chico, CA").
        label = ", ".join(dict.fromkeys(
            x for x in (p.get("name"), p.get("city"), p.get("county"), p.get("state")) if x
        ))
        # ipywidgets.Select keys its options by label, so two identical labels collapse into one.
        seen[label] = seen.get(label, 0) + 1
        if seen[label] > 1:
            label = f"{label} ({seen[label]})"
        lon, lat = f["geometry"]["coordinates"]
        hits.append((label, (lon, lat, p.get("extent"))))
    return hits

def hit_to_view(hit):
    '''Where the camera should land for a geocoder hit.'''
    lon, lat, extent = hit
    if not extent:
        return MapViewState(longitude=lon, latitude=lat, zoom=JUMP_ZOOM)
    minlon, maxlat, maxlon, minlat = extent
    vs = bbox_to_view((minlon, minlat, maxlon, maxlat))
    # Never land below MINZOOM: you asked to go to a place, so you want to see its features, not
    # the row-group boxes. Never land above 16 either, or a small town fills the screen.
    return MapViewState(longitude=vs.longitude, latitude=vs.latitude,
                        zoom=min(max(vs.zoom, MINZOOM), 16.0))

## Viewport &harr; bounding box

Turning the camera into a box to read. This needs to know how wide the map is in pixels, which
lonboard does not report, so it is a constant. Too small and the edges of the map stay empty; too
large and you read features you do not draw. Erring large.

In [ ]:
VIEW_W, VIEW_H = 1400, 700

def _lat_to_y(lat):
    s = math.sin(math.radians(max(min(lat, 85.0), -85.0)))
    return 0.5 - math.log((1 + s) / (1 - s)) / (4 * math.pi)

def _y_to_lat(y):
    return math.degrees(2 * math.atan(math.exp((0.5 - y) * 2 * math.pi)) - math.pi / 2)

def view_to_bbox(vs):
    world = 512 * (2 ** vs.zoom)
    half_lon = 360.0 * VIEW_W / world / 2
    yc, half_y = _lat_to_y(vs.latitude), VIEW_H / world / 2
    return (vs.longitude - half_lon, _y_to_lat(yc + half_y),
            vs.longitude + half_lon, _y_to_lat(yc - half_y))

def bbox_to_view(bbox, pad=1.15):
    xmin, ymin, xmax, ymax = bbox
    dlon = max((xmax - xmin) * pad, 1e-6)
    dy = max(abs(_lat_to_y(ymin) - _lat_to_y(ymax)) * pad, 1e-9)
    zoom = max(0.0, min(18.0, min(math.log2(360.0 * VIEW_W / 512 / dlon),
                                  math.log2(VIEW_H / 512 / dy))))
    return MapViewState(longitude=(xmin + xmax) / 2, latitude=(ymin + ymax) / 2, zoom=zoom)

## The map

Two update paths. A **layer-set change** (tick, untick, reorder, region, geocode) builds a new
`Map`; a **camera move** swaps the layers on the live map in place. Everything runs synchronously
in a browser-event handler (no threads, no timers), the one path Colab, JupyterLab, and VSCode
all deliver reliably.

In [ ]:
OPENING = ["census-acs-housing", "hifld-hospitals"]

# state["built"] maps layer name -> its current lonboard layers. state["bbox"] is the last box
# drawn on the map; read_key / read_bbox record what the last viewport read covered.
state = {"region": "northern-ca", "layers": list(OPENING), "built": {}, "map": None,
         "view": None, "bbox": None,
         "read_key": None, "read_bbox": None}

canvas = W.VBox([])       # holds the map
status = W.HTML()         # status line under the controls
snippet_out = W.VBox([])  # holds the drawn-box query, below the map

def layer_stack():
    '''Layers to draw, bottom to top: the ticked layers, then the AOI outline on top.'''
    return [l for name in state["layers"] for l in state["built"].get(name, [])] + [aoi_layer(state["region"])]

def rebuild():
    """Build a new Map and swap it in. Used for layer-set changes (tick, untick, reorder, region, geocode)."""
    m = Map(layer_stack(), height=650, view_state=state["view"], show_tooltip=True,
            basemap=MaplibreBasemap(style=CartoStyle.DarkMatter))
    m.observe(on_camera, names="view_state")
    m.observe(on_select, names="selected_bounds")
    state["map"] = m
    canvas.children = (m,)

repaint = rebuild

def restack():
    """Swap the layers on the existing map in place. Used for camera moves (pan/zoom)."""
    if state["map"] is None:
        rebuild()
    else:
        state["map"].layers = layer_stack()

READ_PAD = 1.35     # read a little past the viewport, so a small pan needs no new read

def pad_bbox(b, f=READ_PAD):
    dx, dy = (b[2] - b[0]) * (f - 1) / 2, (b[3] - b[1]) * (f - 1) / 2
    return (b[0] - dx, b[1] - dy, b[2] + dx, b[3] + dy)

def covers(outer, inner):
    return (outer[0] <= inner[0] and outer[1] <= inner[1]
            and outer[2] >= inner[2] and outer[3] >= inner[3])

def render(view=None, fresh=True):
    """Read what the selection needs for this viewport and show it. fresh=True rebuilds the map
    (layer-set changes), fresh=False restacks the live one (camera moves)."""
    show = rebuild if fresh else restack
    try:
        region, chosen = state["region"], state["layers"]
        state["view"] = view = view or state["view"] or bbox_to_view(aoi_bbox(region))

        if not chosen:
            state["built"], state["read_key"], state["read_bbox"] = {}, None, None
            show()
            status.value = "Pick a layer."
            return

        # Below MINZOOM, big layers show their row-group boxes instead of every feature.
        boxed = [n for n in chosen if is_big(region, n) and view.zoom < MINZOOM]
        read = [n for n in chosen if n not in boxed]
        bbox = view_to_bbox(view)

        # Nothing to do if this viewport is still inside what the last read covered.
        if not fresh and state["read_key"] == (region, tuple(read), tuple(boxed)) \
                and state["read_bbox"] and covers(state["read_bbox"], bbox):
            return

        status.value = "<b>reading…</b> the read covers your whole viewport. Zoom in for a smaller one."
        # Read every layer at once (obstore and geoarrow release the GIL).
        padded = pad_bbox(bbox)
        with ThreadPoolExecutor(max_workers=max(len(read), 1)) as pool:
            built = dict(zip(read, pool.map(lambda n: build(region, n, padded), read)))
        built.update({n: partition_boxes(region, n) for n in boxed})

        state["built"] = built
        state["read_key"], state["read_bbox"] = (region, tuple(read), tuple(boxed)), padded
        show()

        # Status line: each layer with its row count, in draw order.
        drawn = sum(l.table.num_rows for n in read for l in built[n])
        parts = [f"{sum(l.table.num_rows for l in built[n]):,} {n}" for n in read]
        parts += [f"{sum(l.table.num_rows for l in built[n])} row-group boxes ({n})" for n in boxed]
        line = f"<b>zoom {view.zoom:.1f}</b> · " + " · ".join(parts)

        if boxed:
            line += (f" · <b>zoom past {MINZOOM:.0f}</b> to load "
                     f"{' and '.join(boxed)} for real.")
        elif drawn > HEAVY:
            line += (f" · <b>{drawn:,} features on screen.</b> Zoom in, or untick a layer, "
                     f"for a faster map.")
        status.value = line
    except Exception as exc:
        # Surface errors in the status line: a failure in a comm handler can otherwise go silent.
        status.value = f"<b style='color:#F0E442'>failed:</b> {type(exc).__name__}: {exc}"
        raise

def same_view(a, b):
    return a and b and (round(a.longitude, 6), round(a.latitude, 6), round(a.zoom, 4)) == \
                       (round(b.longitude, 6), round(b.latitude, 6), round(b.zoom, 4))

def on_camera(change):
    """Handle a camera move. The echo check ignores the event the map emits for a view we set."""
    if same_view(change["new"], state["view"]):
        return
    render(change["new"], fresh=False)


# ## Draw a box -> a copyable query
#
# The map's box button (lower-right) draws a bounding box. on_select turns it into a standalone
# snippet for the ticked layers, clipped to that box, and shows it below the map.

import html as _html

def copy_button(code):
    """Copy-to-clipboard button. The copy runs in an inline onclick, since Python cannot reach the
    browser clipboard from the kernel."""
    js = (f"const b = this; navigator.clipboard.writeText({json.dumps(code)})"
          ".then(() => { b.textContent = 'copied ✓';"
          " setTimeout(() => { b.textContent = 'copy snippet'; }, 1200); });")
    return W.HTML(f'<button class="bb-copy" onclick="{_html.escape(js, quote=True)}">copy snippet</button>')

# A real ipywidgets Button (unlike copy) because clearing has to reset state["bbox"] in the kernel.
clear_b = W.Button(description="clear", layout=W.Layout(width="70px"))

def on_clear(_):
    state["bbox"] = None
    if state["map"] is not None:
        state["map"].selected_bounds = None   # clear the map's own rectangle
    snippet_out.children = ()
    status.value = "snippet cleared. Draw another box (box button, lower-right) for a new one."

clear_b.on_click(on_clear)

def show_snippet():
    """Render the query for the last-drawn box below the map, kept in step with the ticked layers."""
    if not state["bbox"]:
        snippet_out.children = ()
        return
    if not state["layers"]:
        snippet_out.children = (W.HTML("<i>Tick at least one layer, then draw a box.</i>"),)
        return
    code = render_snippet(state["region"], state["layers"], state["bbox"])
    header = W.HBox(
        [W.HTML("<b>Query for the box you drew</b>"),
         copy_button(code), clear_b],
        layout=W.Layout(align_items="center"))
    header.add_class("bb-snippet")
    snippet_out.children = (W.HTML(PANEL_CSS),   # each Colab iframe needs its own copy of the CSS
                            header,
                            W.HTML(f'<pre class="bb-code">{_html.escape(code)}</pre>'))

def on_select(change):
    bounds = change["new"]
    if not bounds:                       # a fresh Map resets selected_bounds to None
        return
    state["bbox"] = tuple(bounds)
    show_snippet()
    status.value = ("<b>box drawn.</b> The snippet is below the map: copy it. "
                    "Draw again (box button, lower-right) to update it.")

## The query builder

In [ ]:
# Builds a standalone snippet: DuckDB reads the drawn box off the bucket (the bbox.* comparisons
# prune row groups), materializes it as a table named after the layer, and hands it to lonboard's
# viz(). The read is plain HTTPS: no region, no creds, no signing. One variable per ticked layer;
# point `url` at the one you want to map.

def layer_url(region, name):
    return f"https://data.source.coop/{ROOT}/geoparquet/{region}/{region}-{name}.parquet"

def render_snippet(region, layers, bbox):
    xmin, ymin, xmax, ymax = bbox
    var = {n: f"{region}-{n}".replace("-", "_") for n in layers}
    width = max(len(v) for v in var.values())
    files = "\n".join(f'{var[n]:<{width}} = "{layer_url(region, n)}"' for n in layers)
    first = layers[0]
    count = {2: "two", 3: "three"}.get(len(layers), str(len(layers)))
    pick = (f"      # pick one of the {count} above and fill in here"
            if len(layers) > 1 else "")
    plural = "layers" if len(layers) > 1 else "layer"
    return f'''
import duckdb
from lonboard import viz

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs; INSTALL spatial; LOAD spatial")

# the box you drew (EPSG:4326)
xmin, ymin, xmax, ymax = {xmin:.6f}, {ymin:.6f}, {xmax:.6f}, {ymax:.6f}

# the {plural} you added (variable = filename)
{files}

url  = {var[first]}{pick}
name = url.split("/")[-1].removesuffix(".parquet")       # the filename, e.g. "{region}-{first}"

# CREATE ... AS materializes the clipped box into a table named after the layer, so con.table(name)
# hands it back to viz. The table name is quoted because the filename has dashes.
con.sql(f"""
    CREATE OR REPLACE TABLE '{{name}}' AS
    SELECT geometry::geometry AS geometry, * EXCLUDE (geometry)
    FROM read_parquet('{{url}}')
    WHERE bbox.xmin <= {{xmax}} AND bbox.xmax >= {{xmin}}
      AND bbox.ymin <= {{ymax}} AND bbox.ymax >= {{ymin}}
""")
con.sql(f"FROM '{{name}}'").show()
viz(con.table(name))'''

## Controls, and the map

In [ ]:
# Styling for the control panel. The <style> ships as an HTML widget inside the panel (and the
# snippet gets its own copy) so it reaches the widgets even inside Colab's per-output iframes.
PANEL_CSS = '''
<style>
.bb-panel {
  background: #12151A; border: 1px solid #262C34; border-radius: 6px;
  padding: 14px 16px; margin-bottom: 8px; width: max-content;
  font-family: ui-monospace, SFMono-Regular, Menlo, monospace;
}
.bb-panel .widget-label, .bb-panel label, .bb-panel select, .bb-panel button,
.bb-panel input[type="text"], .bb-panel .widget-html-content {
  font-family: ui-monospace, SFMono-Regular, Menlo, monospace !important;
  font-size: 12px !important; color: #C6CDD6;
}
.bb-panel select, .bb-panel button, .bb-panel input[type="text"] {
  background: #1B2029; border: 1px solid #2E3743; border-radius: 4px; color: #C6CDD6;
}
.bb-panel input[type="text"]::placeholder { color: #5C6672; }
.bb-panel button:hover:enabled { background: #262E3A; }
.bb-panel button:disabled { opacity: 0.4; }
.bb-panel select option:checked { background: #2E3743; }
.bb-head {
  font-size: 10px !important; letter-spacing: 0.12em; text-transform: uppercase;
  color: #6E7A88 !important; margin-bottom: 6px;
}
/* Fixed-height status strip, so changing feature counts do not reflow the panel. */
.bb-status {
  min-height: 20px; margin-top: 10px; padding-top: 8px; border-top: 1px solid #262C34;
  color: #8E99A6 !important;
}
.bb-status b { color: #E6EDF3; font-weight: 600; }
/* Swatch + checkbox as a centred flex row. */
.bb-swatch { display: flex; align-items: center; height: 100%; }
.bb-swatch .widget-html-content { display: flex; align-items: center; line-height: 1; }
.bb-row { align-items: center; }
.bb-row .widget-checkbox { margin: 0; }
/* Snippet header row (label + copy + clear). .bb-copy is a raw <button>; clear is a jupyter-button. */
.bb-snippet { margin: 8px 0 0 0; }
.bb-snippet .widget-html-content {
  font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: 12px; color: #8E99A6;
}
.bb-snippet b { color: #E6EDF3; }
.bb-copy, .bb-snippet .jupyter-button {
  background: #1B2029; border: 1px solid #2E3743; border-radius: 4px; color: #C6CDD6;
  font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: 12px;
  height: 24px; padding: 0 10px; margin-left: 10px; cursor: pointer;
}
.bb-copy:hover, .bb-snippet .jupyter-button:hover { background: #262E3A; }
/* The generated query <pre> below the map. */
.bb-code {
  background: #12151A; border: 1px solid #262C34; border-radius: 6px;
  padding: 12px 14px; margin: 8px 0 0 0; overflow-x: auto; max-width: 920px;
  font-family: ui-monospace, SFMono-Regular, Menlo, monospace; font-size: 12px;
  color: #C6CDD6; line-height: 1.5;
}
</style>
'''

SWATCH_OFF = "#3A424D"
CIVIDIS_CHIP = "linear-gradient(90deg, {}, {}, {})".format(
    *["#%02X%02X%02X" % tuple(int(c) for c in np.array(CIVIDIS(x)[:3]) * 255)
      for x in (0.0, 0.5, 1.0)]
)

def swatch(name, on):
    """Colour chip for a layer row: its flat colour, or the cividis ramp for a choropleth."""
    cfg = LAYERS[name]
    bg = SWATCH_OFF if not on else (CIVIDIS_CHIP if "ramp" in cfg else cfg["color"])
    return f'<div style="width:11px;height:11px;border-radius:2px;background:{bg}"></div>'

# ---- layers ----------------------------------------------------------------------------------
checks, marks, rows = {}, {}, []

def sync_cap():
    """At the cap, disable the unticked boxes. Ticked boxes stay live, so you can trade one out."""
    at_cap = len(state["layers"]) >= MAX_LAYERS
    for name, box in checks.items():
        box.disabled = at_cap and not box.value

def on_check(change):
    name = change["owner"].description
    if change["new"]:
        if len(state["layers"]) >= MAX_LAYERS:
            change["owner"].value = False
            status.value = f"<b>{MAX_LAYERS} layers maximum.</b> Untick one to make room."
            return
        state["layers"].append(name)      # newly ticked: drawn last, so it lands on top
    else:
        state["layers"].remove(name)
    marks[name].value = swatch(name, change["new"])
    sync_cap()
    sync_order(keep=name if change["new"] else None)
    if state["bbox"]:
        show_snippet()   # keep the query in step with what is ticked
    if change["new"]:
        render()          # a newly ticked layer has to be read before it can be drawn
    else:
        state["built"].pop(name, None)
        repaint()         # unticking is just a restack: drop it and repaint, no network

for name in LAYERS:
    on = name in state["layers"]
    checks[name] = W.Checkbox(value=on, description=name, indent=False,
                              layout=W.Layout(width="196px", margin="0", flex="0 0 auto"))
    marks[name] = W.HTML(swatch(name, on),
                         layout=W.Layout(width="19px", margin="0", flex="0 0 auto"))
    marks[name].add_class("bb-swatch")
    checks[name].observe(on_check, names="value")
    row = W.HBox([marks[name], checks[name]],
                 layout=W.Layout(align_items="center", margin="0", height="22px"))
    row.add_class("bb-row")
    rows.append(row)

# ---- region ----------------------------------------------------------------------------------
region_w = W.Dropdown(options=REGIONS, value=state["region"], description="",
                      layout=W.Layout(width="215px"))

def on_region(change):
    state["region"] = change["new"]
    search_w.value = ""                   # the old region's hits are meaningless here
    state["bbox"] = None                  # a box drawn in the old region is meaningless here
    show_snippet()
    show_hits(())
    render(bbox_to_view(aoi_bbox(change["new"])))

region_w.observe(on_region, names="value")

# ---- find a place ----------------------------------------------------------------------------
search_w = W.Text(placeholder="town, county, address… then Enter", continuous_update=False,
                  description="", layout=W.Layout(width="215px"))
hits_w = W.Select(options=(), rows=6, description="",
                  layout=W.Layout(width="215px", display="none", margin="6px 0 0 0"))

def show_hits(hits):
    """Fill the results list, or hide it when empty. Mute the value observer across the swap so
    setting .options does not fire a camera jump."""
    hits_w.unobserve(on_hit, names="value")
    hits_w.options = tuple(hits)
    hits_w.value = None
    hits_w.observe(on_hit, names="value")
    hits_w.layout.display = "flex" if hits else "none"

def on_search(change):
    """Runs once per search: the box syncs its value on Enter (continuous_update=False)."""
    query = change["new"].strip()
    if len(query) < 3:                    # 1-2 letters match half the country
        show_hits(())
        return
    try:
        hits = photon(query, aoi_bbox(state["region"]))
    except Exception as exc:              # a geocoder outage must not take the map down
        show_hits(())
        status.value = f"geocoder unavailable ({type(exc).__name__}). The map still works."
        return
    show_hits(hits)
    if not hits:
        status.value = f"no match for <b>{query}</b> in {state['region']}."

def on_hit(change):
    if change["new"] is None:
        return
    render(hit_to_view(change["new"]))    # jump straight to the place

search_w.observe(on_search, names="value")
hits_w.observe(on_hit, names="value")

# ---- draw order ------------------------------------------------------------------------------
# state["layers"] is draw order (first = underneath). The widget shows the reverse, so the top of
# the list is the top layer. Every conversion between the two happens here.
order_w = W.Select(options=tuple(reversed(state["layers"])), rows=6, description="",
                   layout=W.Layout(width="215px"))
fwd_b = W.Button(description="▲ bring forward", layout=W.Layout(width="105px", margin="6px 4px 0 0"))
back_b = W.Button(description="▼ send back", layout=W.Layout(width="105px", margin="6px 0 0 0"))

def sync_order(keep=None):
    """Rebuild the order list from state["layers"], keeping the current selection. Mute the observer
    across the swap so setting .options does not move the highlight."""
    keep = keep if keep is not None else order_w.value
    order_w.unobserve(sync_buttons, names="value")
    order_w.options = tuple(reversed(state["layers"]))
    order_w.value = keep if keep in state["layers"] else None
    order_w.observe(sync_buttons, names="value")
    sync_buttons()

def sync_buttons(*_):
    sel, layers = order_w.value, state["layers"]
    fwd_b.disabled = sel is None or layers.index(sel) == len(layers) - 1
    back_b.disabled = sel is None or layers.index(sel) == 0

def move(step):
    """Move the selected layer in draw order (+1 = toward the top, -1 = toward the bottom). No
    network: the layers are already built, so this just reshuffles and repaints."""
    def handler(_):
        layers, sel = state["layers"], order_w.value
        if sel is None:
            return
        i, j = layers.index(sel), layers.index(sel) + step
        if not 0 <= j < len(layers):
            return
        layers[i], layers[j] = layers[j], layers[i]
        sync_order(keep=sel)      # keep it selected: you usually click again
        repaint()
    return handler

order_w.observe(sync_buttons, names="value")
fwd_b.on_click(move(+1))          # forward on the map = later in the draw order
back_b.on_click(move(-1))

# ---- panel -----------------------------------------------------------------------------------
def column(title, *widgets):
    return W.VBox([W.HTML(f'<div class="bb-head">{title}</div>'), *widgets],
                  layout=W.Layout(margin="0 28px 0 0"))

panel = W.VBox([
    W.HTML(PANEL_CSS),                    # the CSS lives inside the panel's own (i)frame
    W.HBox([
        W.VBox([
            column("Set region", region_w),
            column("Find a place", search_w, hits_w),
        ], layout=W.Layout(margin="0 28px 0 0")),
        column("Layers", W.HTML(f'<div class="bb-note">Limit {MAX_LAYERS} layers</div>'), *rows),
        column("Draw order (top = on top)", order_w, W.HBox([fwd_b, back_b])),
    ]),
    status,
])
panel.add_class("bb-panel")
status.add_class("bb-status")

sync_cap()
sync_order()
render()    # build the opening map; the next cell displays it

In [ ]:
# Display-only cell (kept separate so Colab renders it reliably).
#
# Draw a box on the map with the ▢ button (lower-right): click two corners and a copy-pasteable
# query for the ticked layers, clipped to that box, appears below the map.
display(panel, canvas, snippet_out)